# Part 2 v5 — Two-Stage Model + GLM + Blend (Refined)

v2 base + data cleaning only. No new features.

**v5 changes:**
- Data cleaning: occupation cleanup, mode imputation, campaign_code dropped
- Categorical fix: vehicle_value_group, flood_zone → string (NOT year_built)
- No new features (written_premium is circular/leakage, others are noise)

**v2 baseline**: Blend D² avg 0.0672 (motor 0.0489, buildings 0.0699, contents 0.0827)

In [0]:
# Step 1: Load data, XGBoost v2 results, and enrichment
import pandas as pd
import numpy as np
import os, pickle

%pip install statsmodels openpyxl -q

DATA_DIR = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/data_emerging/'
PROC_DIR = os.path.join(DATA_DIR, 'proc')

# Load processed training data
train = {}
for line in ['motor', 'buildings', 'contents']:
    train[line] = pd.read_parquet(os.path.join(PROC_DIR, f'proc_{line}.parquet'))
    print(f'{line}: {train[line].shape}, target mean={train[line]["ultimate_claims"].mean():.4f}')

# === v5 DATA CLEANING ===
CATEGORICAL_FIX = ['vehicle_value_group', 'engine_size_cc', 'branch_id',
                    'flood_zone', 'customer_since_year']  # NOT year_built
MODE_IMPUTE = ['overnight_parking', 'occupation', 'marital_status', 'soil_type',
               'roof_type', 'energy_rating', 'smoke_alarms', 'security_level']

def clean_occupation(s):
    if pd.isna(s):
        return s
    s = str(s).strip().lower()
    if s.startswith('self-employed '):
        s = s.replace('self-employed ', '')
    return s

def clean_data(df):
    df = df.copy()
    if 'campaign_code' in df.columns:
        df.drop('campaign_code', axis=1, inplace=True)
    if 'occupation' in df.columns:
        df['occupation'] = df['occupation'].apply(clean_occupation)
    for col in CATEGORICAL_FIX:
        if col in df.columns:
            df[col] = df[col].astype(str)
    for col in MODE_IMPUTE:
        if col in df.columns and df[col].isna().any():
            mode_val = df[col].mode()
            if len(mode_val) > 0:
                df[col] = df[col].fillna(mode_val[0])
    return df

for line in ['motor', 'buildings', 'contents']:
    train[line] = clean_data(train[line])
    print(f'{line} v5 cleaned: {train[line].shape}')

# Load test files and enrichment
test = {}
for line in ['motor', 'buildings', 'contents']:
    test[line] = pd.read_parquet(os.path.join(DATA_DIR, f'test_{line}.parquet'))
customers = pd.read_parquet(os.path.join(DATA_DIR, 'customers.parquet'))
addresses = pd.read_parquet(os.path.join(DATA_DIR, 'addresses.parquet'))
weather = pd.read_parquet(os.path.join(DATA_DIR, 'weather.parquet'))
sample_submission = pd.read_parquet(os.path.join(DATA_DIR, 'sample_submission.parquet'))

# Load XGBoost v5 refined results
with open(os.path.join(PROC_DIR, 'xgboost_v5_refined_results.pkl'), 'rb') as f:
    xgb_v2 = pickle.load(f)

xgb_oof = xgb_v2['oof_predictions']
xgb_cv_scores = xgb_v2['cv_scores']
xgb_test_preds = xgb_v2['test_predictions']
TREND_FACTORS = xgb_v2['trend_factors']
FEATURE_SETS = xgb_v2['feature_sets']

# Load v1 results for comparison
with open(os.path.join(PROC_DIR, 'xgboost_v1_results.pkl'), 'rb') as f:
    v1_results = pickle.load(f)

V1_BLEND_D2 = {'motor': 0.0431, 'buildings': 0.0719, 'contents': 0.0684}

print(f'\nXGBoost v2 CV D2: {xgb_cv_scores}')
print(f'Trend factors: {TREND_FACTORS}')
print(f'Feature sets: {FEATURE_SETS}')

In [0]:
# Step 2: Prepare test data (clean + join enrichment)
AGE_BAND_MAP = {'17-24': 20.5, '25-34': 29.5, '35-49': 42.0, '50-64': 57.0, '65+': 70.0}

for line in ['motor', 'buildings', 'contents']:
    df = test[line]
    ps = pd.to_datetime(df['period_start'])
    pe = pd.to_datetime(df['period_end'])
    df['exposure'] = ((pe - ps).dt.days + 1) / 365.0
    df['Year'] = ps.dt.year
    for col in ['policy_id', 'customer_id', 'address_id', 'row_id']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
    for col in df.columns:
        if col.endswith('_age_band') or col == 'age_band':
            new_col = col.replace('_age_band', '_age')
            df[new_col] = df[col].map(AGE_BAND_MAP)
    if 'compulsory_excess' in df.columns and df['compulsory_excess'].nunique() <= 1:
        df.drop('compulsory_excess', axis=1, inplace=True)

cust_lens = customers['customer_id'].str.len().value_counts()
if cust_lens.nunique() > 1:
    max_len = cust_lens.index.max()
    customers['customer_id'] = customers['customer_id'].str.zfill(max_len)
for line in ['motor', 'buildings', 'contents']:
    df = test[line]
    if 'customer_id' in df.columns:
        lens = df['customer_id'].str.len().value_counts()
        if lens.nunique() > 1:
            max_len = lens.index.max()
            df['customer_id'] = df['customer_id'].str.zfill(max_len)

customers_clean = customers.drop(columns=['surname', 'date_of_birth'], errors='ignore')
weather_agg = weather.groupby(['address_id', 'year'], as_index=False).mean(numeric_only=True)
weather_agg.rename(columns={'year': 'Year'}, inplace=True)

for line in ['motor', 'buildings', 'contents']:
    df = test[line]
    df = df.merge(customers_clean, on='customer_id', how='left', suffixes=('', '_cust'))
    df = df.merge(addresses, on='address_id', how='left', suffixes=('', '_addr'))
    df = df.merge(weather_agg, on=['address_id', 'Year'], how='left', suffixes=('', '_weather'))
    test[line] = df
    print(f'test_{line}: {df.shape}')

# Apply v5 cleaning to test data (after enrichment)
for line in ['motor', 'buildings', 'contents']:
    test[line] = clean_data(test[line])
    print(f'test_{line} v5 cleaned: {test[line].shape}')

In [0]:
# Step 3: Two-stage model — Logistic P(claim) × Gamma severity
# Separates the zero-inflation problem from severity estimation
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

SMOOTHING = 50
LP_CLIP = 20

def tweedie_deviance(y, y_pred, p=1.5):
    y = np.array(y, dtype=float)
    y_pred = np.maximum(y_pred, 1e-20)
    return 2 * (np.power(np.maximum(y, 0), 2-p) / ((1-p)*(2-p))
                - y * np.power(y_pred, 1-p) / (1-p)
                + np.power(y_pred, 2-p) / (2-p))

def tweedie_d2(y, y_pred, p=1.5):
    dev = tweedie_deviance(y, y_pred, p).sum()
    dev_null = tweedie_deviance(y, np.mean(y), p).sum()
    return 1 - dev / dev_null

def target_encode(train_series, train_target, apply_series, smoothing=50):
    prior = train_target.mean()
    grp = pd.DataFrame({'cat': train_series, 'target': train_target})
    stats = grp.groupby('cat')['target'].agg(['mean', 'count'])
    smooth = (stats['mean'] * stats['count'] + prior * smoothing) / (stats['count'] + smoothing)
    train_enc = train_series.map(smooth).fillna(prior)
    apply_enc = apply_series.map(smooth).fillna(prior)
    return train_enc, apply_enc

N_FOLDS = 5
two_stage_cv_scores = {}
two_stage_oof = {}
two_stage_test_preds = {}

for line in ['motor', 'buildings', 'contents']:
    print(f'\n{"="*60}')
    print(f'=== {line.upper()} Two-Stage Model ===')
    print(f'{"="*60}')

    train_df = train[line]
    test_df = test[line]
    features = FEATURE_SETS[line]
    y_all = train_df['ultimate_claims'].values
    has_claim = (y_all > 0).astype(int)
    exp_all = train_df['exposure'].values
    exp_test = test_df['exposure'].values
    n_claims = has_claim.sum()
    print(f'  Claims: {n_claims}/{len(has_claim)} ({has_claim.mean():.1%})')
    print(f'  Mean severity (claims only): {y_all[y_all > 0].mean():.2f}')

    # --- k-fold CV ---
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    oof_pred = np.zeros(len(train_df))

    for fold, (tr_idx, val_idx) in enumerate(kf.split(train_df)):
        tr_df = train_df.iloc[tr_idx]
        val_df = train_df.iloc[val_idx]
        y_tr = y_all[tr_idx]
        y_val = y_all[val_idx]
        has_claim_tr = has_claim[tr_idx]
        exp_tr = exp_all[tr_idx]
        exp_val = exp_all[val_idx]

        # Prepare features: target encode categoricals, standardize numerics
        X_tr = pd.DataFrame(index=tr_idx)
        X_val = pd.DataFrame(index=val_idx)
        for col in features:
            tr_vals = tr_df[col]
            val_vals = val_df[col]
            is_cat = tr_vals.dtype == 'object' or tr_vals.dtype == 'bool'
            if is_cat:
                tr_enc, val_enc = target_encode(tr_vals, y_tr, val_vals, SMOOTHING)
                X_tr[col] = tr_enc.values
                X_val[col] = val_enc.values
            else:
                med = tr_vals.median()
                X_tr[col] = tr_vals.fillna(med).values
                X_val[col] = val_vals.fillna(med).values

        scaler = StandardScaler()
        X_tr_s = sm.add_constant(scaler.fit_transform(X_tr), has_constant='add')
        X_val_s = sm.add_constant(scaler.transform(X_val), has_constant='add')

        # Stage 1: Logistic regression for P(claim)
        try:
            logit = sm.GLM(has_claim_tr, X_tr_s, family=sm.families.Binomial(),
                          offset=np.log(exp_tr + 1e-10))
            logit_res = logit.fit(max_iter=100)
            p_claim = logit_res.predict(X_val_s)
            p_claim = np.clip(p_claim, 1e-6, 1 - 1e-6)
        except Exception as e:
            print(f'  Fold {fold+1}: Stage 1 FAILED ({e}), using base rate')
            p_claim = np.full(len(val_idx), has_claim_tr.mean())

        # Stage 2: Gamma GLM for severity (claims rows only)
        claims_mask = has_claim_tr > 0
        if claims_mask.sum() < 20:
            print(f'  Fold {fold+1}: too few claims ({claims_mask.sum()}), mean severity')
            severity = np.full(len(val_idx), y_tr[claims_mask].mean() if claims_mask.sum() > 0 else y_all.mean())
        else:
            try:
                gamma = sm.GLM(y_tr[claims_mask], X_tr_s[claims_mask],
                              family=sm.families.Gamma(link=sm.families.links.Log()))
                gamma_res = gamma.fit(max_iter=100)
                eta = np.clip(np.dot(X_val_s, gamma_res.params), -LP_CLIP, LP_CLIP)
                severity = np.maximum(np.exp(eta), 0.01)
            except Exception as e:
                print(f'  Fold {fold+1}: Stage 2 FAILED ({e}), mean severity')
                severity = np.full(len(val_idx), y_tr[claims_mask].mean())

        # Combine: P(claim) * severity
        val_pred = p_claim * severity
        oof_pred[val_idx] = val_pred
        d2 = tweedie_d2(y_val, val_pred, p=1.5)
        print(f'  Fold {fold+1}: D2 = {d2:.4f}, pred mean = {val_pred.mean():.1f} (actual {y_val.mean():.1f}), '
              f'P(claim) = {p_claim.mean():.4f}, severity = {severity.mean():.1f}')

    two_stage_oof[line] = oof_pred
    two_stage_cv_scores[line] = tweedie_d2(y_all, oof_pred, p=1.5)
    print(f'  Two-stage OOF D2: {two_stage_cv_scores[line]:.4f}')
    print(f'  OOF pred mean: {oof_pred.mean():.4f} (actual: {y_all.mean():.4f})')

    # --- Final model on all data for test predictions ---
    X_all_enc = pd.DataFrame(index=train_df.index)
    X_test_enc = pd.DataFrame(index=test_df.index)
    for col in features:
        tr_vals = train_df[col]
        test_vals = test_df[col]
        is_cat = tr_vals.dtype == 'object' or tr_vals.dtype == 'bool'
        if is_cat:
            tr_enc, test_enc = target_encode(tr_vals, y_all, test_vals, SMOOTHING)
            X_all_enc[col] = tr_enc.values
            X_test_enc[col] = test_enc.values
        else:
            med = tr_vals.median()
            X_all_enc[col] = tr_vals.fillna(med).values
            X_test_enc[col] = test_vals.fillna(med).values

    scaler = StandardScaler()
    X_all_s = sm.add_constant(scaler.fit_transform(X_all_enc), has_constant='add')
    X_test_s = sm.add_constant(scaler.transform(X_test_enc), has_constant='add')

    # Stage 1: Logistic on all data
    logit = sm.GLM(has_claim, X_all_s, family=sm.families.Binomial(),
                   offset=np.log(exp_all + 1e-10))
    logit_res = logit.fit(max_iter=100)
    p_claim_test = np.clip(logit_res.predict(X_test_s), 1e-6, 1 - 1e-6)

    # Stage 2: Gamma on all claims
    claims_mask = has_claim > 0
    gamma = sm.GLM(y_all[claims_mask], X_all_s[claims_mask],
                   family=sm.families.Gamma(link=sm.families.links.Log()))
    gamma_res = gamma.fit(max_iter=100)
    eta_test = np.clip(np.dot(X_test_s, gamma_res.params), -LP_CLIP, LP_CLIP)
    severity_test = np.maximum(np.exp(eta_test), 0.01)

    test_pred = p_claim_test * severity_test
    two_stage_test_preds[line] = test_pred
    print(f'  Test: mean={test_pred.mean():.4f}, P(claim) mean={p_claim_test.mean():.4f}, severity mean={severity_test.mean():.4f}')

print(f'\n{"="*60}')
print(f'=== Two-Stage Summary ===')
for line in ['motor', 'buildings', 'contents']:
    print(f'  {line}: two-stage D2={two_stage_cv_scores[line]:.4f} | XGB v2={xgb_cv_scores[line]:.4f}')

In [0]:
# Step 4: Tweedie GLM v2 with curated features (same as v1 but 10-13 features instead of 20)

glm_cv_scores = {}
glm_oof_preds = {}
glm_test_preds = {}
glm_coefs = {}

for line in ['motor', 'buildings', 'contents']:
    print(f'\n{"="*60}')
    print(f'=== {line.upper()} GLM v2 (curated features) ===')
    print(f'{"="*60}')

    train_df = train[line]
    test_df = test[line]
    features = FEATURE_SETS[line]
    y_all = train_df['ultimate_claims'].values
    exp_all = train_df['exposure'].values
    exp_test = test_df['exposure'].values
    print(f'  Features: {len(features)}')

    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    fold_scores = []
    oof_pred = np.zeros(len(train_df))

    for fold, (tr_idx, val_idx) in enumerate(kf.split(train_df)):
        y_tr, y_val = y_all[tr_idx], y_all[val_idx]
        exp_tr, exp_val = exp_all[tr_idx], exp_all[val_idx]
        tr_df, val_df = train_df.iloc[tr_idx], train_df.iloc[val_idx]

        X_tr = pd.DataFrame(index=tr_idx)
        X_val = pd.DataFrame(index=val_idx)
        for col in features:
            tr_vals, val_vals = tr_df[col], val_df[col]
            is_cat = tr_vals.dtype == 'object' or tr_vals.dtype == 'bool'
            if is_cat:
                tr_enc, val_enc = target_encode(tr_vals, y_tr, val_vals, SMOOTHING)
                X_tr[col] = tr_enc.values
                X_val[col] = val_enc.values
            else:
                med = tr_vals.median()
                X_tr[col] = tr_vals.fillna(med).values
                X_val[col] = val_vals.fillna(med).values

        scaler = StandardScaler()
        X_tr_s = pd.DataFrame(scaler.fit_transform(X_tr), columns=X_tr.columns, index=X_tr.index)
        X_tr_s = sm.add_constant(X_tr_s, has_constant='add')
        X_val_s = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)
        X_val_s = sm.add_constant(X_val_s, has_constant='add')

        try:
            model = sm.GLM(y_tr, X_tr_s,
                           family=sm.families.Tweedie(link=sm.families.links.Log(), var_power=1.5),
                           offset=np.log(exp_tr + 1e-10))
            result = model.fit(max_iter=100)
            eta = np.clip(np.dot(X_val_s.values, result.params), -LP_CLIP, LP_CLIP)
            val_pred = np.maximum(np.exp(eta + np.log(exp_val + 1e-10)), 0)
            d2 = tweedie_d2(y_val, val_pred, p=1.5)
            fold_scores.append(d2)
            oof_pred[val_idx] = val_pred
            print(f'  Fold {fold+1}: D2 = {d2:.4f}, pred mean = {val_pred.mean():.1f} (actual {y_val.mean():.1f})')
        except Exception as e:
            print(f'  Fold {fold+1}: FAILED ({e})')
            fold_scores.append(0)
            oof_pred[val_idx] = exp_val * y_tr.mean()

    glm_cv_scores[line] = np.mean(fold_scores)
    glm_oof_preds[line] = oof_pred
    print(f'  CV mean D2: {glm_cv_scores[line]:.4f} (+/- {np.std(fold_scores):.4f})')
    print(f'  (XGB v2: {xgb_cv_scores[line]:.4f}, two-stage: {two_stage_cv_scores[line]:.4f})')

    # Final model on all data
    X_all_enc = pd.DataFrame(index=train_df.index)
    X_test_enc = pd.DataFrame(index=test_df.index)
    for col in features:
        tr_vals, test_vals = train_df[col], test_df[col]
        is_cat = tr_vals.dtype == 'object' or tr_vals.dtype == 'bool'
        if is_cat:
            tr_enc, test_enc = target_encode(tr_vals, y_all, test_vals, SMOOTHING)
            X_all_enc[col] = tr_enc.values
            X_test_enc[col] = test_enc.values
        else:
            med = tr_vals.median()
            X_all_enc[col] = tr_vals.fillna(med).values
            X_test_enc[col] = test_vals.fillna(med).values

    scaler = StandardScaler()
    X_all_s = pd.DataFrame(scaler.fit_transform(X_all_enc), columns=X_all_enc.columns, index=X_all_enc.index)
    X_all_s = sm.add_constant(X_all_s, has_constant='add')
    X_test_s = pd.DataFrame(scaler.transform(X_test_enc), columns=X_test_enc.columns, index=X_test_enc.index)
    X_test_s = sm.add_constant(X_test_s, has_constant='add')

    model = sm.GLM(y_all, X_all_s,
                   family=sm.families.Tweedie(link=sm.families.links.Log(), var_power=1.5),
                   offset=np.log(exp_all + 1e-10))
    result = model.fit(max_iter=100)
    glm_coefs[line] = pd.DataFrame({'feature': X_all_s.columns, 'coef': result.params, 'pvalue': result.pvalues})

    eta_test = np.clip(np.dot(X_test_s.values, result.params), -LP_CLIP, LP_CLIP)
    test_pred = np.maximum(np.exp(eta_test + np.log(exp_test + 1e-10)), 0.01)
    glm_test_preds[line] = test_pred
    print(f'  Test predictions: mean={test_pred.mean():.4f}')

In [0]:
# Step 5: 3-way blend (XGB v2 + GLM v2 + two-stage), calibrate, generate submissions
SUB_DIR = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/submissions_emerging_v5_refined/'
os.makedirs(SUB_DIR, exist_ok=True)

# Optimize 3-way blend weights on OOF predictions
blend_d2 = {}
best_weights = {}

for line in ['motor', 'buildings', 'contents']:
    y_all = train[line]['ultimate_claims'].values
    best_d2 = -999
    best_w = (0, 0, 1)

    for wx in np.arange(0, 0.81, 0.05):
        for wg in np.arange(0, 0.81 - wx, 0.05):
            wt = 1.0 - wx - wg
            blend = wx * xgb_oof[line] + wg * glm_oof_preds[line] + wt * two_stage_oof[line]
            d2 = tweedie_d2(y_all, blend, p=1.5)
            if d2 > best_d2:
                best_d2 = d2
                best_w = (wx, wg, wt)

    blend_d2[line] = best_d2
    best_weights[line] = best_w
    print(f'{line}: XGB={best_w[0]:.2f}, GLM={best_w[1]:.2f}, TwoStage={best_w[2]:.2f} -> D2={best_d2:.4f}')

print(f'\nAverage blend D2: {np.mean(list(blend_d2.values())):.4f}')
print(f'(v1 blend D2 avg: {np.mean(list(V1_BLEND_D2.values())):.4f})')

# Per-line calibration
CAL_TARGETS = {line: train[line]['ultimate_claims'].mean() for line in ['motor', 'buildings', 'contents']}

# Generate calibrated blended submission
submission = sample_submission.copy()
for line in ['motor', 'buildings', 'contents']:
    test_df = test[line]
    wx, wg, wt = best_weights[line]

    xgb_p = xgb_test_preds[line] * TREND_FACTORS[line]
    glm_p = np.clip(np.nan_to_num(glm_test_preds[line], posinf=50000, neginf=0.01), 0.01, 50000) * TREND_FACTORS[line]
    ts_p = np.clip(two_stage_test_preds[line], 0.01, 50000) * TREND_FACTORS[line]

    blend_pred = wx * xgb_p + wg * glm_p + wt * ts_p
    blend_pred = np.clip(blend_pred, 0.01, 50000)

    cal_target = CAL_TARGETS[line] * TREND_FACTORS[line]
    cal_factor = cal_target / blend_pred.mean()
    blend_pred = np.clip(blend_pred * cal_factor, 0.01, 50000)

    print(f'{line}: blend mean={blend_pred.mean():.4f} (target={cal_target:.4f}, cal_factor={cal_factor:.4f})')

    pred_map = dict(zip(test_df['row_id'], blend_pred))
    mask = submission['row_id'].isin(pred_map)
    submission.loc[mask, 'prediction'] = submission.loc[mask, 'row_id'].map(pred_map)

submission['prediction'] = submission['prediction'].clip(lower=0.01, upper=50000)
submission['prediction'] = submission['prediction'].replace([np.inf, -np.inf], 50000).fillna(0.01)
sub_path = os.path.join(SUB_DIR, 'blend_calibrated_v5_refined.csv')
submission.to_csv(sub_path, index=False)
print(f'\nSaved: {sub_path} ({len(submission)} rows, mean={submission["prediction"].mean():.4f})')

# Also save two-stage-only and GLM-only calibrated submissions
for model_name, preds in [('two_stage', two_stage_test_preds), ('glm', glm_test_preds)]:
    sub = sample_submission.copy()
    for line in ['motor', 'buildings', 'contents']:
        test_df = test[line]
        pred = np.clip(preds[line], 0.01, 50000) * TREND_FACTORS[line]
        cal = CAL_TARGETS[line] * TREND_FACTORS[line] / pred.mean()
        pred = np.clip(pred * cal, 0.01, 50000)
        pred_map = dict(zip(test_df['row_id'], pred))
        mask = sub['row_id'].isin(pred_map)
        sub.loc[mask, 'prediction'] = sub.loc[mask, 'row_id'].map(pred_map)
    sub['prediction'] = sub['prediction'].clip(lower=0.01, upper=50000).fillna(0.01)
    sub.to_csv(os.path.join(SUB_DIR, f'{model_name}_calibrated_v5_refined.csv'), index=False)
    print(f'{model_name} calibrated: mean={sub["prediction"].mean():.4f}')

In [0]:
# Step 6: Excel checkpoint — save all model comparison and results
EXCEL_DIR = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/Excel files_emerging_v5_refined/'
os.makedirs(EXCEL_DIR, exist_ok=True)

with pd.ExcelWriter(os.path.join(EXCEL_DIR, 'v5_refined_results.xlsx'), engine='openpyxl') as writer:
    comparison = pd.DataFrame([
        {'line': line,
         'xgb_v1_d2': v1_results['cv_scores'][line],
         'xgb_v2_d2': xgb_cv_scores[line],
         'glm_v2_d2': glm_cv_scores[line],
         'two_stage_d2': two_stage_cv_scores[line],
         'blend_v2_d2': blend_d2[line],
         'blend_v1_d2': V1_BLEND_D2[line],
         'w_xgb': best_weights[line][0],
         'w_glm': best_weights[line][1],
         'w_twostage': best_weights[line][2],
         'n_features': len(FEATURE_SETS[line])}
        for line in ['motor', 'buildings', 'contents']
    ])
    comparison.to_excel(writer, sheet_name='Model Comparison', index=False)
    for line in ['motor', 'buildings', 'contents']:
        glm_coefs[line].to_excel(writer, sheet_name=f'{line} GLM coefs', index=False)

print(f'Saved: {EXCEL_DIR}v5_refined_results.xlsx')
print(f'\n=== Final v5 Refined Model Comparison ===')
print(comparison.to_string(index=False))
print(f'\nAverage D2:')
print(f'  XGB v1:     {comparison["xgb_v1_d2"].mean():.4f}')
print(f'  XGB v2:     {comparison["xgb_v2_d2"].mean():.4f}')
print(f'  GLM v2:     {comparison["glm_v2_d2"].mean():.4f}')
print(f'  Two-stage:  {comparison["two_stage_d2"].mean():.4f}')
print(f'  Blend v2:  {comparison["blend_v2_d2"].mean():.4f}')
print(f'  Blend v1:  {comparison["blend_v1_d2"].mean():.4f}')
print(f'\nSubmissions in {SUB_DIR}:')
for f in sorted(os.listdir(SUB_DIR)):
    print(f'  {f}')